In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns
from epiweeks import Week
import matplotlib.pyplot as plt

In [44]:
state = 'BA'
df_climate = pd.read_parquet(f'/Users/eduardoaraujo/Documents/Github/load_infodengue_data/data/climate/{state}_climate_new.parquet', columns = ['geocode', 'temp_med', 'umid_med', 'precip_tot'])
#df_climate = add_epiweek_label(df_climate)
df_climate.reset_index(inplace=  True)
#df_climate['date'] = pd.to_datetime(df_climate['date'])
#df_climate['season'] = df_climate['date'].apply(get_season)
df_climate.head()

,date,geocode,temp_med,umid_med,precip_tot
0,2022-06-23,2905305,20.0051,77.4814,1.9906
1,2022-06-21,2900355,22.1360,85.3025,15.0788
2,2022-06-22,2900355,22.7616,80.8354,8.3320
3,2022-06-23,2900355,22.6841,85.7385,16.0693
4,2022-06-24,2900355,22.8539,79.8645,19.4382


In [45]:
df_climate.loc[(df_climate.geocode=='2916104')]

,date,geocode,temp_med,umid_med,precip_tot
1808,2022-05-09,2916104,NaN,NaN,0.0
1815,2022-05-10,2916104,NaN,NaN,0.0
8155,2022-05-01,2916104,NaN,NaN,0.0
8156,2022-05-02,2916104,NaN,NaN,0.0
8157,2022-05-03,2916104,NaN,NaN,0.0
...,...,...,...,...,...
2271236,2022-03-29,2916104,NaN,NaN,0.0
2271237,2022-03-30,2916104,NaN,NaN,0.0
2271238,2022-03-31,2916104,NaN,NaN,0.0
2275184,2021-07-26,2916104,NaN,NaN,0.0


In [3]:
df_climate.loc[df_climate.precip_tot > 10].shape[0]/df_climate.shape[0]

0.675426011078052

In [4]:
df_climate.precip_tot.max()

777.0263

In [5]:
df_climate.head()

,geocode,temp_med,umid_med,precip_tot
date,,,,
2023-07-29,1300904,28.9224,55.6624,0.0051
2023-07-30,1300904,28.3629,53.3801,0.0017
2023-07-31,1300904,29.0945,54.2962,0.0053
2023-07-01,1301001,25.6020,77.3595,0.0057
2023-07-02,1301001,26.1782,82.1402,0.0731


In [27]:
def get_season(date):
    month = date.month
    day = date.day
    if (month == 12 and day >= 21) or (month in [1, 2]) or (month == 3 and day <= 20):
        return 'Summer'
    elif (month == 3 and day >= 21) or (month in [4, 5]) or (month == 6 and day <= 20):
        return 'Autumn'
    elif (month == 6 and day >= 21) or (month in [7, 8]) or (month == 9 and day <= 20):
        return 'Winter'
    elif (month == 9 and day >= 21) or (month in [10, 11]) or (month == 12 and day <= 20):
        return 'Spring'
        
def add_epiweek_label(df_w):
    '''
    This function assumes that the dataframe has a datetime index
    and add the epiweek and year value
    '''

    df_w['epiweek_label'] = [Week.fromdate(x) for x in df_w.index]

    df_w['epiweek_label'] = df_w['epiweek_label'].astype(str)

    df_w['epiweek'] = df_w['epiweek_label'].astype(str).str[-2:].astype(int)
    
    df_w['year'] = df_w['epiweek_label'].astype(str).str[:4].astype(int)

    return df_w


def load_agg_clima(state, ini_date = '2010-01-01', end_date ='2020-12-31'):
    '''
    This function load and aggregates the climatic variables by the epidemiological year
    '''
    df_climate = pd.read_parquet(f'/Users/eduardoaraujo/Documents/Github/load_infodengue_data/data/climate/{state}_climate_new.parquet', columns = ['geocode', 'temp_med', 'umid_med', 'precip_tot'])

    df_climate['geocode'] = df_climate['geocode'].astype(int)
    df_climate.index = pd.to_datetime(df_climate.index)
        
    df_climate = df_climate.loc[(df_climate.index >= ini_date) & (df_climate.index<= end_date)].sort_index()

    df_climate['thr_temp'] = 0

    df_climate.loc[df_climate.temp_med > 20, 'thr_temp'] = 1

    df_climate['thr_umid'] = 0

    df_climate.loc[df_climate.umid_med > 60, 'thr_umid'] = 1

    df_climate['thr_prec'] = 0

    df_climate.loc[df_climate.precip_tot > 10, 'thr_prec'] = 1
    
    df_climate = add_epiweek_label(df_climate)

    df_climate['thr_temp_umid'] = 0

    df_climate.loc[(df_climate.temp_med > 20) & (df_climate.umid_med > 60), 'thr_temp_umid'] = 1
    
    df_clima_agg = pd.DataFrame()
    
    for year in df_climate.year.unique():
    
        filter1 = (df_climate.year == year) & (df_climate.epiweek >= 41)
        filter2 = (df_climate.year == year+1) & (df_climate.epiweek < 41)
        
        df_ = df_climate.loc[filter1 | filter2].reset_index()
        
        df_['season'] = df_['date'].apply(get_season)
        
        df_ = df_.groupby(['geocode', 'season']).agg({'temp_med': 'mean', 'umid_med': 'mean', 'precip_tot': 'sum',
                                                                                       'thr_temp':'sum',
                                                                                       'thr_umid':'sum', 
                                                                                       'thr_prec':'sum',
                                                                                       'thr_temp_umid': 'sum'}).reset_index()
    
        df_['year'] = year
        
        df_clima_agg = pd.concat([df_clima_agg, df_])
    
    return df_clima_agg    

In [28]:
states_BR = ['AL',
 'BA',
 'CE',
 'MA',
 'PB',
 'PE',
 'PI',
 'SE',
 'RN',
 'SP',
 'MG',
 'RJ',
 'ES',
 'AM',
 'AP',
 'TO',
 'RR',
 'RO',
 'AC',
 'PA',
 'DF',
 'GO',
 'MT',
 'MS',
 'RS',
 'SC',
 'PR']


### Clima no período 2010_2017: 

In [29]:
state = 'SC'
df_ = load_agg_clima(state)
df_ = df_.loc[df_.year.isin(np.arange(2010, 2017))]
df_.year.unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016])

In [30]:
df_

,geocode,season,temp_med,umid_med,precip_tot,thr_temp,thr_umid,thr_prec,thr_temp_umid,year
0,4200051,Autumn,15.092976,82.883416,1140.7943,6,92,27,6,2010
1,4200051,Spring,17.484473,77.961088,1864.8297,18,90,39,18,2010
2,4200051,Summer,20.926321,82.789012,1635.4629,69,90,49,69,2010
3,4200051,Winter,12.665523,82.160973,3495.9868,0,92,46,0,2010
4,4200101,Autumn,15.755782,81.093460,1483.4119,9,92,26,9,2010
...,...,...,...,...,...,...,...,...,...,...
1175,4219853,Winter,16.210432,71.322096,612.6590,16,85,11,10,2016
1176,4220000,Autumn,19.599274,80.570023,2178.4782,44,91,46,44,2016
1177,4220000,Spring,20.714163,76.724340,1221.8481,57,86,30,56,2016
1178,4220000,Summer,25.318596,79.605736,1357.0509,90,90,26,90,2016


In [31]:
df_.year.unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016])

In [32]:
df_perfis = pd.read_csv('../Dados/Dengue/dengue_pattern_10_16.csv', sep = ';', usecols = ['muni_code', 'dengue_pattern'])

df_perfis.head()

,muni_code,dengue_pattern
0,1100015,Episódico/Epidêmico
1,1100023,Epidêmico
2,1100031,Episódico/Epidêmico
3,1100049,Epidêmico
4,1100056,Epidêmico


The cell below concatenate the aggregated dataframes of all states in just one:

In [33]:
df_climate_all = pd.DataFrame()

for state in states_BR: 
    
    df_climate = load_agg_clima(state, ini_date = '2010-01-01', end_date = '2017-12-31')

    df_climate = df_climate.loc[df_climate.year.isin(np.arange(2010, 2017))]
    
    df_climate_all = pd.concat([df_climate_all, df_climate])

# merge the climatic dataset with the dengue patterns 
df_climate_all = df_climate_all.merge(df_perfis, left_on = 'geocode', right_on = 'muni_code').drop('muni_code', axis =1)
    
df_climate_all.head()

,geocode,season,temp_med,umid_med,precip_tot,thr_temp,thr_umid,thr_prec,thr_temp_umid,year,dengue_pattern
0,2700102,Autumn,24.499337,75.563752,950.0245,92,86,22,86,2010,Episódico/Epidêmico
1,2700102,Spring,26.110110,63.122558,452.0930,90,55,9,55,2010,Episódico/Epidêmico
2,2700102,Summer,26.810854,64.194899,621.8873,90,59,20,59,2010,Episódico/Epidêmico
3,2700102,Winter,22.189745,74.240779,534.1305,92,91,15,91,2010,Episódico/Epidêmico
4,2700201,Autumn,24.157386,84.502892,1865.3530,92,92,57,92,2010,Episódico/Epidêmico


Verifying the presence of null values in the data: 

In [34]:
df_climate_all.isnull().sum()

geocode            0
season             0
temp_med          84
umid_med          84
precip_tot         0
thr_temp           0
thr_umid           0
thr_prec           0
thr_temp_umid      0
year               0
dengue_pattern     0
dtype: int64

Save the dataframe: 

In [35]:
%%time
df_climate_all.to_csv('../Dados/Determinantes/clima/clima_agg_10_16_season.csv')

CPU times: user 544 ms, sys: 26.7 ms, total: 571 ms
Wall time: 607 ms


### Clima no período 2020_2022:

In [36]:
state = 'AC'
df_ = load_agg_clima(state, ini_date = '2017-01-01', end_date = '2023-12-31')
df_ = df_.loc[df_.year.isin(np.arange(2017, 2023))]
df_.year.unique()

array([2017, 2018, 2019, 2020, 2021, 2022])

In [37]:
df_perfis = pd.read_csv('../Dados/Dengue/dengue_pattern_17_22.csv', sep = ';', usecols = ['muni_code', 'dengue_pattern'])

df_perfis.head()

,muni_code,dengue_pattern
0,1100015,Epidêmico
1,1100023,Epidêmico
2,1100031,Episódico/Epidêmico
3,1100049,Epidêmico
4,1100056,Episódico/Epidêmico


In [38]:
df_climate_all = pd.DataFrame()

for state in states_BR: 
    
    df_climate = load_agg_clima(state, ini_date = '2017-01-01', end_date = '2023-12-31')

    df_climate = df_climate.loc[df_climate.year.isin(np.arange(2017, 2023))]
    
    df_climate_all = pd.concat([df_climate_all, df_climate])

# merge the climatic dataset with the dengue patterns 
df_climate_all = df_climate_all.merge(df_perfis, left_on = 'geocode', right_on = 'muni_code').drop('muni_code', axis = 1)
    
df_climate_all.head()

,geocode,season,temp_med,umid_med,precip_tot,thr_temp,thr_umid,thr_prec,thr_temp_umid,year,dengue_pattern
0,2700102,Autumn,24.936441,71.559392,528.5750,92,88,12,88,2017,Episódico/Epidêmico
1,2700102,Spring,26.661764,59.088583,104.5703,90,41,1,41,2017,Episódico/Epidêmico
2,2700102,Summer,27.669137,60.282593,390.1183,90,42,11,42,2017,Episódico/Epidêmico
3,2700102,Winter,23.509128,67.164870,217.3068,92,86,2,86,2017,Episódico/Epidêmico
4,2700201,Autumn,24.229479,83.254857,1000.2414,92,92,43,92,2017,Episódico/Epidêmico


In [39]:
df_climate_all.isnull().sum()

geocode            0
season             0
temp_med          72
umid_med          72
precip_tot         0
thr_temp           0
thr_umid           0
thr_prec           0
thr_temp_umid      0
year               0
dengue_pattern     0
dtype: int64

In [40]:
df_climate_all.to_csv('../Dados/Determinantes/clima/clima_agg_17_22_season.csv')